# Session 4: From a messy inbox to executive charts

**Business request:** Combine five months of expense files from four departments, clean their inconsistent labels, take an audit sample, summarize approved spending, and export English and Korean reports.

> Run this notebook from the repository root. The data is synthetic.

## Before we get started

1. Add the following python package to your environment (to export notebooks to HTML):
   - `notebook`

    In this class, we use UV for Python management, Python packages are installed with uv add xxx

2. Download the `inbox.zip` dataset from iCampus and save it in the `data/inbox` directory.
   - check if your data folder is being ignored by `.gitignore`

## 1. Discover and list the files in the messy inbox
- More information about the pathlib module: https://medium.com/data-science/why-you-should-start-using-pathlib-as-an-alternative-to-the-os-module-d9eccd994745

In [5]:
## Let's define the location of the data, output, and temporary directories so that we don't need to hardcode and repeat paths throughout the code
from pathlib import Path

data_dir = Path("data/inbox")
output_dir = Path("outputs")
temp_dir = Path("temp")

## Then, let's create the output and temporary directories if they don't already exist

output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(parents=True, exist_ok=True)

In [6]:
## Then, let's list all files in the data directory
## show Path(data_dir).iterdir() first for demonstration

list(data_dir.iterdir())

all_files_glob = [str(path) for path in Path(data_dir).iterdir()]

print(all_files_glob)

['data\\inbox\\2026-01_hr_expenses.csv', 'data\\inbox\\2026-01_marketing_expenses.xlsx', 'data\\inbox\\2026-01_operations_expenses.xlsx', 'data\\inbox\\2026-01_sales_expenses.csv', 'data\\inbox\\2026-02_hr_expenses.xlsx', 'data\\inbox\\2026-02_marketing_expenses.csv', 'data\\inbox\\2026-02_operations_expenses.csv', 'data\\inbox\\2026-02_sales_expenses.xlsx', 'data\\inbox\\2026-03_hr_expenses.csv', 'data\\inbox\\2026-03_marketing_expenses.xlsx', 'data\\inbox\\2026-03_operations_expenses.xlsx', 'data\\inbox\\2026-03_sales_expenses.csv', 'data\\inbox\\2026-04_hr_expenses.xlsx', 'data\\inbox\\2026-04_marketing_expenses.csv', 'data\\inbox\\2026-04_operations_expenses.csv', 'data\\inbox\\2026-04_sales_expenses.xlsx', 'data\\inbox\\2026-05_hr_expenses.csv', 'data\\inbox\\2026-05_marketing_expenses.xlsx', 'data\\inbox\\2026-05_operations_expenses.xlsx', 'data\\inbox\\2026-05_sales_expenses.csv', 'data\\inbox\\DATA_DICTIONARY.md']


In [12]:
all_files_glob

['data\\inbox\\2026-01_hr_expenses.csv',
 'data\\inbox\\2026-01_marketing_expenses.xlsx',
 'data\\inbox\\2026-01_operations_expenses.xlsx',
 'data\\inbox\\2026-01_sales_expenses.csv',
 'data\\inbox\\2026-02_hr_expenses.xlsx',
 'data\\inbox\\2026-02_marketing_expenses.csv',
 'data\\inbox\\2026-02_operations_expenses.csv',
 'data\\inbox\\2026-02_sales_expenses.xlsx',
 'data\\inbox\\2026-03_hr_expenses.csv',
 'data\\inbox\\2026-03_marketing_expenses.xlsx',
 'data\\inbox\\2026-03_operations_expenses.xlsx',
 'data\\inbox\\2026-03_sales_expenses.csv',
 'data\\inbox\\2026-04_hr_expenses.xlsx',
 'data\\inbox\\2026-04_marketing_expenses.csv',
 'data\\inbox\\2026-04_operations_expenses.csv',
 'data\\inbox\\2026-04_sales_expenses.xlsx',
 'data\\inbox\\2026-05_hr_expenses.csv',
 'data\\inbox\\2026-05_marketing_expenses.xlsx',
 'data\\inbox\\2026-05_operations_expenses.xlsx',
 'data\\inbox\\2026-05_sales_expenses.csv',
 'data\\inbox\\DATA_DICTIONARY.md']

In [25]:
# Filter all file paths in the all_files_glob list to include only CSV and Excel files

data_files = [f for f in all_files_glob if f.endswith('.csv') or f.endswith('.xlsx')]

data_files


['data\\inbox\\2026-01_hr_expenses.csv',
 'data\\inbox\\2026-01_marketing_expenses.xlsx',
 'data\\inbox\\2026-01_operations_expenses.xlsx',
 'data\\inbox\\2026-01_sales_expenses.csv',
 'data\\inbox\\2026-02_hr_expenses.xlsx',
 'data\\inbox\\2026-02_marketing_expenses.csv',
 'data\\inbox\\2026-02_operations_expenses.csv',
 'data\\inbox\\2026-02_sales_expenses.xlsx',
 'data\\inbox\\2026-03_hr_expenses.csv',
 'data\\inbox\\2026-03_marketing_expenses.xlsx',
 'data\\inbox\\2026-03_operations_expenses.xlsx',
 'data\\inbox\\2026-03_sales_expenses.csv',
 'data\\inbox\\2026-04_hr_expenses.xlsx',
 'data\\inbox\\2026-04_marketing_expenses.csv',
 'data\\inbox\\2026-04_operations_expenses.csv',
 'data\\inbox\\2026-04_sales_expenses.xlsx',
 'data\\inbox\\2026-05_hr_expenses.csv',
 'data\\inbox\\2026-05_marketing_expenses.xlsx',
 'data\\inbox\\2026-05_operations_expenses.xlsx',
 'data\\inbox\\2026-05_sales_expenses.csv']

In [27]:
## Create DataFrame of Files to get a structured view of all CSV and Excel files
import pandas as pd

files_df = pd.DataFrame({
    'filename': data_files,
    'filetype': ['csv' if f.endswith('.csv') else 'xlsx' for f in data_files]
})

files_df.head(10)

,filename,filetype
0,data\inbox\2026-01_hr_expenses.csv,csv
1,data\inbox\2026-01_marketing_expenses.xlsx,xlsx
2,data\inbox\2026-01_operations_expenses.xlsx,xlsx
3,data\inbox\2026-01_sales_expenses.csv,csv
4,data\inbox\2026-02_hr_expenses.xlsx,xlsx
5,data\inbox\2026-02_marketing_expenses.csv,csv
6,data\inbox\2026-02_operations_expenses.csv,csv
7,data\inbox\2026-02_sales_expenses.xlsx,xlsx
8,data\inbox\2026-03_hr_expenses.csv,csv
9,data\inbox\2026-03_marketing_expenses.xlsx,xlsx


In [29]:
## Lets create two lists for each filetype 

csv_files = [f for f in data_files if f.endswith('.csv')]
xlsx_files = [f for f in data_files if f.endswith('.xlsx')]

print("CSV Files:", csv_files)
print("Excel Files:", xlsx_files)

CSV Files: ['data\\inbox\\2026-01_hr_expenses.csv', 'data\\inbox\\2026-01_sales_expenses.csv', 'data\\inbox\\2026-02_marketing_expenses.csv', 'data\\inbox\\2026-02_operations_expenses.csv', 'data\\inbox\\2026-03_hr_expenses.csv', 'data\\inbox\\2026-03_sales_expenses.csv', 'data\\inbox\\2026-04_marketing_expenses.csv', 'data\\inbox\\2026-04_operations_expenses.csv', 'data\\inbox\\2026-05_hr_expenses.csv', 'data\\inbox\\2026-05_sales_expenses.csv']
Excel Files: ['data\\inbox\\2026-01_marketing_expenses.xlsx', 'data\\inbox\\2026-01_operations_expenses.xlsx', 'data\\inbox\\2026-02_hr_expenses.xlsx', 'data\\inbox\\2026-02_sales_expenses.xlsx', 'data\\inbox\\2026-03_marketing_expenses.xlsx', 'data\\inbox\\2026-03_operations_expenses.xlsx', 'data\\inbox\\2026-04_hr_expenses.xlsx', 'data\\inbox\\2026-04_sales_expenses.xlsx', 'data\\inbox\\2026-05_marketing_expenses.xlsx', 'data\\inbox\\2026-05_operations_expenses.xlsx']


In [45]:
## Let's print a preview of some CSV files. What is the same? What is different?

csv_preview = pd.read_csv(csv_files[8]) # Try some more by changing the index
csv_preview.head()

,ID,Expense Date,Dept,Cost Category,Merchant,Total,Approved?,Staff ID,Note
0,EX-202605-HR-01,2026-05-18,인사,Meals,Office Cafe,"220,000",승인,H168,Meals expense
1,EX-202605-HR-02,2026-05-09,People,Hiring,Saramin,"2,283,000",승인,H172,Recruiting expense
2,EX-202605-HR-03,2026-05-02,인사,Training,Korea Management Institute,235000,승인,H173,Training expense
3,EX-202605-HR-04,2026-05-19,HR,채용비,Saramin,"1,710,000",승인,H177,Recruiting expense
4,EX-202605-HR-05,2026-05-09,HR,Hiring,Saramin,2095000,Y,H170,Recruiting expense


In [47]:
## Let's do the same for the Excel files

excel_preview = pd.read_excel(xlsx_files[4])
excel_preview.head()

,Transaction ID,Date,Department,Category,Vendor,Amount KRW,Status,Employee ID,Description
0,EX-202603-MA-01,2026-03-12,MKT,광고비,Naver Ads,1899000,Rejected,M110,Advertising expense
1,EX-202603-MA-02,2026-03-15,마케팅,소프트웨어,Notion,440000,승인,M102,Software expense
2,EX-202603-MA-03,2026-03-04,MKT,광고비,Kakao Business,1576000,Approved,M115,Advertising expense
3,EX-202603-MA-04,2026-03-21,마케팅,Software,Notion,940000,Approved,M101,Software expense
4,EX-202603-MA-05,2026-03-12,MKT,소프트웨어,DataCloud Korea,438000,승인,M104,Software expense


## 2. Merging the DataFrames

`pandas.concat()` function concatenate two or more pandas objects like DataFrames or Series along a particular axis. It is especially useful when combining datasets either vertically (row-wise) or horizontally (column-wise). Read about using conact to append dataframes in pandas: [Future Replacement of the Append Method in Panda Python](https://www.geeksforgeeks.org/python/future-replacement-of-the-append-method-in-panda-python/)

In [48]:
## Append all CSV files into a single DataFrame and then inspect it
csv_combined_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

print(f"Combined shape: {csv_combined_df.shape}")
csv_combined_df.head()

Combined shape: (227, 36)


,ID,Expense Date,Dept,Cost Category,Merchant,Total,Approved?,Staff ID,Note,expense_id,...,Description,전표번호,사용일,부서,항목,거래처,금액(원),승인상태,사번,내역
0,EX-202601-HR-01,2026-01-13,HR,채용비,Wanted,2003000,Rejected,H163,Recruiting expense,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EX-202601-HR-02,2026-01-02,HR,Training,Korea Management Institute,753000,Y,H164,Training expense,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EX-202601-HR-03,2026-01-24,HR,Meals,Local Restaurant,207000,Approved,H161,Meals expense,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,EX-202601-HR-04,2026-01-02,People,교육비,Coursera,1056000,검토중,H177,Training expense,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,EX-202601-HR-05,2026-01-25,HR,교육비,Korea Management Institute,"490,000",Approved,H160,Training expense,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
## Check all the columns in the combined DataFrame
csv_combined_df.columns

Index(['ID', 'Expense Date', 'Dept', 'Cost Category', 'Merchant', 'Total',
       'Approved?', 'Staff ID', 'Note', 'expense_id', 'transaction_date',
       'team', 'expense_type', 'supplier', 'amount', 'approval', 'employee',
       'memo', 'Transaction ID', 'Date', 'Department', 'Category', 'Vendor',
       'Amount KRW', 'Status', 'Employee ID', 'Description', '전표번호', '사용일',
       '부서', '항목', '거래처', '금액(원)', '승인상태', '사번', '내역'],
      dtype='str')

You can find the column mapping for the cell below at https://github.com/pimatskku/BAM5103/blob/main/column_mapping.txt

In [49]:
## Build a mapping table from each department's column names to the canonical schema
# Canonical fields: transaction_id, expense_date, department, category, vendor, amount_krw, approval_status, employee_id, description

column_mapping = {
    # HR convention
    "ID": "transaction_id",
    "Expense Date": "expense_date",
    "Dept": "department",
    "Cost Category": "category",
    "Merchant": "vendor",
    "Total": "amount_krw",
    "Approved?": "approval_status",
    "Staff ID": "employee_id",
    "Note": "description",
    # Sales convention
    "expense_id": "transaction_id",
    "transaction_date": "expense_date",
    "team": "department",
    "expense_type": "category",
    "supplier": "vendor",
    "amount": "amount_krw",
    "approval": "approval_status",
    "employee": "employee_id",
    "memo": "description",
    # Marketing convention
    "Transaction ID": "transaction_id",
    "Date": "expense_date",
    "Department": "department",
    "Category": "category",
    "Vendor": "vendor",
    "Amount KRW": "amount_krw",
    "Status": "approval_status",
    "Employee ID": "employee_id",
    "Description": "description",
    # Operations convention (Korean)
    "전표번호": "transaction_id",
    "사용일": "expense_date",
    "부서": "department",
    "항목": "category",
    "거래처": "vendor",
    "금액(원)": "amount_krw",
    "승인상태": "approval_status",
    "사번": "employee_id",
    "내역": "description",
}

In [52]:
## Read each CSV, rename its columns to the canonical schema, add a source column then combine them all
renamed_csv_dataframes = []

for f in csv_files:
    df = pd.read_csv(f)
    df = df.rename(columns=column_mapping)
    df['source'] = f.split('/')[-1]
    renamed_csv_dataframes.append(df)

csv_combined_df = pd.concat(renamed_csv_dataframes, ignore_index=True)
print(f"Combined CSV shape: {csv_combined_df.shape}")

Combined CSV shape: (227, 10)


In [53]:
## Let's add the Excel files to the combined DataFrame as well
renamed_excel_dataframes = []

for f in xlsx_files:
    df = pd.read_excel(f)
    df = df.rename(columns=column_mapping)
    df['source'] = f.split('/')[-1]
    renamed_excel_dataframes.append(df)

excel_combined_df = pd.concat(renamed_excel_dataframes, ignore_index=True)

print(f"Combined Excel shape: {excel_combined_df.shape}")

all_combined_df = pd.concat([csv_combined_df, excel_combined_df], ignore_index=True)
print(f"All combined shape: {all_combined_df.shape}")


Combined Excel shape: (221, 10)
All combined shape: (448, 10)


## 3. Get a sample from the DataFrame

Pandas `DataFrame.sample()` function is used to select randomly rows or columns from a DataFrame. It proves particularly helpful while dealing with huge datasets where we want to test or analyze a small representative subset. We can define the number or proportion of items to sample and manage randomness through parameters such as n, frac and random_state. [Read more...](https://www.geeksforgeeks.org/python/python-pandas-dataframe-sample/)

In [75]:
## Lets discover the newly merged dataframe by sampling a few rows. What should be fixed next?

all_combined_df.sample(10)

,transaction_id,expense_date,department,category,vendor,amount_krw,approval_status,employee_id,description,source
297,EX-202602-SA-08,2026-02-25 00:00:00,Sales,Food,Local Restaurant,211000.0,Approved,S137,Meals expense,data\inbox\2026-02_sales_expenses.xlsx
253,EX-202601-OP-06,2026-01-10 00:00:00,Operations,L&D,Fast Campus,1889000.0,Approved,O150,Training expense,data\inbox\2026-01_operations_expenses.xlsx
108,EX-202603-HR-17,2026-03-17,인사,Recruiting,JobKorea,818000,Pending,H175,Recruiting expense,data\inbox\2026-03_hr_expenses.csv
133,EX-202603-SA-22,03/24/2026,Sales,출장비,Kakao T,228000,Pending,S121,Travel expense,data\inbox\2026-03_sales_expenses.csv
115,EX-202603-SA-04,03/07/2026,SLS,Client Events,COEX,1034000,Y,S129,Client Events expense,data\inbox\2026-03_sales_expenses.csv
200,EX-202605-HR-19,2026-05-22,People,L&D,Fast Campus,826000,Y,H174,Training expense,data\inbox\2026-05_hr_expenses.csv
426,EX-202605-OP-01,2026-05-19 00:00:00,운영,Logistics,Quick Service Seoul,1097000.0,Y,O148,Logistics expense,data\inbox\2026-05_operations_expenses.xlsx
203,EX-202605-HR-22,2026-05-26,HR,식비,Office Cafe,"33,000",Y,H163,Meals expense,data\inbox\2026-05_hr_expenses.csv
185,EX-202605-HR-04,2026-05-19,HR,채용비,Saramin,"1,710,000",승인,H177,Recruiting expense,data\inbox\2026-05_hr_expenses.csv
264,EX-202601-OP-17,2026-01-27 00:00:00,Operations,SaaS,Notion,130000.0,승인,O157,Software expense,data\inbox\2026-01_operations_expenses.xlsx


https://github.com/pimatskku/BAM5103/blob/main/value_mappings.txt

In [76]:
## Paste value mappings from https://github.com/pimatskku/BAM5103/blob/main/value_mappings.txt below

department_mapping = {
    "Marketing": "Marketing", "MKT": "Marketing", "마케팅": "Marketing",
    "Sales": "Sales", "SLS": "Sales", "영업": "Sales",
    "Operations": "Operations", "OPS": "Operations", "운영": "Operations",
    "HR": "HR", "People": "HR", "인사": "HR",
}

category_mapping = {
    "Advertising": "Advertising", "Ads": "Advertising", "광고비": "Advertising",
    "Software": "Software", "SaaS": "Software", "소프트웨어": "Software",
    "Travel": "Travel", "Business Travel": "Travel", "출장비": "Travel",
    "Meals": "Meals", "Food": "Meals", "식비": "Meals",
    "Client Events": "Client Events", "Events": "Client Events", "고객행사": "Client Events",
    "Office Supplies": "Office Supplies", "Supplies": "Office Supplies", "사무용품": "Office Supplies",
    "Logistics": "Logistics", "Delivery": "Logistics", "물류비": "Logistics",
    "Training": "Training", "L&D": "Training", "교육비": "Training",
    "Recruiting": "Recruiting", "Hiring": "Recruiting", "채용비": "Recruiting",
}

status_mapping = {
    "Approved": "Approved", "Y": "Approved", "승인": "Approved",
    "Pending": "Pending", "Review": "Pending", "검토중": "Pending",
    "Rejected": "Rejected", "N": "Rejected", "반려": "Rejected",
}

## 4. Transforming (mapping) the values in the dataset with our mapping dictionaries

Pandas is a widely used library for manipulating datasets. There are various in-built functions of pandas, one such function is `pandas.map()`, which is used to map values from two series having one similar column. For mapping two series, the last column of the first should be the same as the index column of the second series, also the values should be unique. [Read more...](https://www.geeksforgeeks.org/python/python-pandas-map/)

In [77]:
## Before we map the values in the dataframe, we create a copy to preserve the original dataframe
transformed_df = all_combined_df.copy()

In [78]:
## Let's transform the values in the dataframe using the mapping dictionaries we created
transformed_df['department'] = transformed_df['department'].map(department_mapping)
transformed_df['category'] = transformed_df['category'].map(category_mapping)
transformed_df['approval_status'] = transformed_df['approval_status'].map(status_mapping)

In [94]:
## Let's check the first few rows of the transformed dataframe to ensure the mappings were applied correctly
transformed_df.sample(10)

,transaction_id,expense_date,department,category,vendor,amount_krw,approval_status,employee_id,description,source
254,EX-202601-OP-07,2026-01-12 00:00:00,Operations,Office Supplies,Coupang Business,488000.0,Rejected,O154,Office Supplies expense,data\inbox\2026-01_operations_expenses.xlsx
54,EX-202602-MA-11,2026-02-04,Marketing,Travel,Kakao T,360000,Approved,M114,NaN,data\inbox\2026-02_marketing_expenses.csv
290,EX-202602-SA-01,2026-02-18 00:00:00,Sales,Travel,Kakao T,45000.0,Approved,S137,Travel expense,data\inbox\2026-02_sales_expenses.xlsx
169,EX-202604-OP-14,2026-04-04,Operations,Software,Notion,"752,000",Rejected,O149,Software expense,data\inbox\2026-04_operations_expenses.csv
412,EX-202605-MA-11,2026-05-25 00:00:00,Marketing,Travel,Korean Air,737000.0,Approved,M109,Travel expense,data\inbox\2026-05_marketing_expenses.xlsx
233,EX-202601-MA-07,2026-01-25 00:00:00,Marketing,Meals,Baemin,57000.0,Approved,M115,Meals expense,data\inbox\2026-01_marketing_expenses.xlsx
147,EX-202604-MA-12,2026-04-16,Marketing,Advertising,Kakao Business,2364000,Approved,M117,NaN,data\inbox\2026-04_marketing_expenses.csv
3,EX-202601-HR-04,2026-01-02,HR,Training,Coursera,1056000,Pending,H177,Training expense,data\inbox\2026-01_hr_expenses.csv
428,EX-202605-OP-03,2026-05-02 00:00:00,Operations,Logistics,Lotte Global Logistics,1198000.0,Approved,O151,Logistics expense,data\inbox\2026-05_operations_expenses.xlsx
231,EX-202601-MA-05,2026-01-27 00:00:00,Marketing,Meals,Baemin,56000.0,Pending,M109,Meals expense,data\inbox\2026-01_marketing_expenses.xlsx


In [97]:
transformed_df['category'].unique()

<StringArray>
[     'Recruiting',        'Training',           'Meals', 'Office Supplies',
          'Travel',        'Software',   'Client Events',     'Advertising',
       'Logistics']
Length: 9, dtype: str

In [102]:
## Export to Excel to have a easier view of the data

transformed_df.to_excel(temp_dir / "combined_data.xlsx")

In [103]:
## A little trick to save Excel files with a timestamp

## Use the timestamp in the filename when exporting to Excel
import datetime
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
transformed_df.to_excel(temp_dir / f"data_{timestamp}.xlsx", index=False)

## Add your temp folder to .gitignore to avoid committing temporary files

### Converting Columns to Datetime

`pandas.to_datetime()` converts argument(s) to datetime. This function is essential for working with date and time data, especially when parsing strings or timestamps into Python's datetime64 format used in Pandas. [Read more](https://www.geeksforgeeks.org/pandas/python-pandas-to_datetime/)

In [104]:
## Let's convert the expense_date column to datetime format
# The "mixed" format allows pandas to infer the datetime format for each entry individually. This is useful when the column contains multiple date formats.
# If a date cannot be parsed, it will be set to NaT (Not a Time) due to errors="coerce".

transformed_df["expense_date"] = pd.to_datetime(transformed_df["expense_date"], format="mixed", errors="coerce")

In [107]:
transformed_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   transaction_id   448 non-null    str           
 1   expense_date     448 non-null    datetime64[us]
 2   department       448 non-null    str           
 3   category         448 non-null    str           
 4   vendor           448 non-null    str           
 5   amount_krw       447 non-null    object        
 6   approval_status  448 non-null    str           
 7   employee_id      448 non-null    str           
 8   description      420 non-null    str           
 9   source           448 non-null    str           
dtypes: datetime64[us](1), object(1), str(8)
memory usage: 35.1+ KB


In [108]:
## And add at 'month' column to transformed_df 
transformed_df['month'] = transformed_df['expense_date'].dt.month

In [115]:
## Let's convert the amount_krw column to numeric values
transformed_df['amount_krw'] = pd.to_numeric(
    transformed_df["amount_krw"].astype("string").str.replace(",", "", regex=False),
    errors="coerce",
)

In [114]:
transformed_df.head(4)

,transaction_id,expense_date,department,category,vendor,amount_krw,approval_status,employee_id,description,source,month
0,EX-202601-HR-01,2026-01-13,HR,Recruiting,Wanted,2003000.0,Rejected,H163,Recruiting expense,data\inbox\2026-01_hr_expenses.csv,1
1,EX-202601-HR-02,2026-01-02,HR,Training,Korea Management Institute,753000.0,Approved,H164,Training expense,data\inbox\2026-01_hr_expenses.csv,1
2,EX-202601-HR-03,2026-01-24,HR,Meals,Local Restaurant,207000.0,Approved,H161,Meals expense,data\inbox\2026-01_hr_expenses.csv,1
3,EX-202601-HR-04,2026-01-02,HR,Training,Coursera,1056000.0,Pending,H177,Training expense,data\inbox\2026-01_hr_expenses.csv,1


In [116]:
# Less find out if there are duplicates in the transaction_id column

transformed_df.sample(10)

,transaction_id,expense_date,department,category,vendor,amount_krw,approval_status,employee_id,description,source,month
338,EX-202603-OP-07,2026-03-22,Operations,Office Supplies,Alpha Stationery,199000.0,Approved,O157,Office Supplies expense,data\inbox\2026-03_operations_expenses.xlsx,3
150,EX-202604-MA-15,2026-04-11,Marketing,Travel,KORAIL,635000.0,Approved,M111,Travel expense,data\inbox\2026-04_marketing_expenses.csv,4
78,EX-202602-OP-08,2026-02-22,Operations,Logistics,Quick Service Seoul,593000.0,Approved,O148,Logistics expense,data\inbox\2026-02_operations_expenses.csv,2
380,EX-202604-HR-27,2026-04-13,HR,Training,Fast Campus,378000.0,Approved,H162,Training expense,data\inbox\2026-04_hr_expenses.xlsx,4
421,EX-202605-MA-20,2026-05-05,Marketing,Meals,Local Restaurant,116000.0,Approved,M111,Meals expense,data\inbox\2026-05_marketing_expenses.xlsx,5
326,EX-202603-MA-17,2026-03-15,Marketing,Meals,Baemin,85000.0,Approved,M107,Meals expense,data\inbox\2026-03_marketing_expenses.xlsx,3
5,EX-202601-HR-06,2026-01-27,HR,Recruiting,JobKorea,478000.0,Approved,H177,Recruiting expense,data\inbox\2026-01_hr_expenses.csv,1
159,EX-202604-OP-04,2026-04-17,Operations,Office Supplies,Alpha Stationery,28000.0,Approved,O144,Office Supplies expense,data\inbox\2026-04_operations_expenses.csv,4
170,EX-202604-OP-15,2026-04-01,Operations,Training,Coursera,769000.0,Approved,O153,NaN,data\inbox\2026-04_operations_expenses.csv,4
168,EX-202604-OP-13,2026-04-25,Operations,Software,Slack,148000.0,Pending,O157,Software expense,data\inbox\2026-04_operations_expenses.csv,4


In [121]:
transformed_df.info()
transformed_df.duplicated("transaction_id").sum()

<class 'pandas.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   transaction_id   448 non-null    str           
 1   expense_date     448 non-null    datetime64[us]
 2   department       448 non-null    str           
 3   category         448 non-null    str           
 4   vendor           448 non-null    str           
 5   amount_krw       447 non-null    Float64       
 6   approval_status  448 non-null    str           
 7   employee_id      448 non-null    str           
 8   description      420 non-null    str           
 9   source           448 non-null    str           
 10  month            448 non-null    int32         
dtypes: Float64(1), datetime64[us](1), int32(1), str(8)
memory usage: 37.3 KB


np.int64(0)

## 5. Quality checks

In [122]:
print("Date range:", transformed_df["expense_date"].min(), "to", transformed_df["expense_date"].max())
print("Duplicate IDs:", transformed_df["transaction_id"].duplicated().sum())
print("\nMissing values:")
(transformed_df.isna().sum().sort_values(ascending=False))

for column in ["department", "category", "approval_status"]:
    print(column, sorted(transformed_df[column].dropna().unique()))

transformed_df.loc[transformed_df["amount_krw"].isna()]

Date range: 2026-01-01 00:00:00 to 2026-05-27 00:00:00
Duplicate IDs: 0

Missing values:
department ['HR', 'Marketing', 'Operations', 'Sales']
category ['Advertising', 'Client Events', 'Logistics', 'Meals', 'Office Supplies', 'Recruiting', 'Software', 'Training', 'Travel']
approval_status ['Approved', 'Pending', 'Rejected']


,transaction_id,expense_date,department,category,vendor,amount_krw,approval_status,employee_id,description,source,month
431,EX-202605-OP-06,2026-05-10,Operations,Logistics,Quick Service Seoul,<NA>,Approved,O148,Logistics expense,data\inbox\2026-05_operations_expenses.xlsx,5


In [123]:
## Create an excel file with all the records that contain a missing value so that this can be sent back to the data provider
transformed_df[transformed_df.isnull().any(axis=1)].to_excel("outputs/missing_values.xlsx", index=False)

## 6. Create a reproducable audit sample

In [133]:
audit_sample = transformed_df.sample(n=100, random_state=50)
transformed_df.sort_values(by='amount_krw',ascending=False)

#print(audit_sample.head())
print(largest_expenses[["transaction_id", "department", "category", "amount_krw"]])

audit_sample.to_excel(temp_dir / "audit_sample.xlsx", index=False)

      transaction_id department       category  amount_krw
139  EX-202604-MA-04  Marketing       Software   8900000.0
414  EX-202605-MA-13  Marketing    Advertising   4100000.0
227  EX-202601-MA-01  Marketing    Advertising   3772000.0
241  EX-202601-MA-15  Marketing    Advertising   3767000.0
325  EX-202603-MA-16  Marketing    Advertising   3686000.0
66   EX-202602-MA-23  Marketing    Advertising   3512000.0
400  EX-202604-SA-19      Sales  Client Events   3496000.0
61   EX-202602-MA-18  Marketing    Advertising   3482000.0
292  EX-202602-SA-03      Sales  Client Events   3481000.0
121  EX-202603-SA-10      Sales  Client Events   3465000.0


## 7. Filter, group, and aggregate

Create two summaries:
- One monthy summary (total approved spending per month, sorted by month)
- One summary of spendings grouped by department and category, sorted by total amount spent

In [137]:
## Create a copy of the dataframe with only approved transactions
approved = transformed_df[transformed_df["approval_status"] == True].copy()

monthly_summary = (
    approved.groupby("month", as_index=False)["amount_krw"]
    .sum()
    .sort_values("month")
)

department_summary = (
    approved.groupby(["department", "category"], as_index=False)["amount_krw"]
    .sum()
    .sort_values("amount_krw", ascending=False)
)

print(monthly_summary)
print(department_summary.head(10))

Empty DataFrame
Columns: [month, amount_krw]
Index: []
Empty DataFrame
Columns: [department, category, amount_krw]
Index: []


## 8. Make presentation-ready English charts

Seaborn is a Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics. The overall package not only has a nice aesthetic quality, but it provides meaningful insights to us as well. [Read more](https://www.geeksforgeeks.org/python/seaborn-lineplot-method-in-python/)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(data=monthly_summary, x="month", y="amount_krw", marker="o", linewidth=2.5, ax=ax)
ax.set_xticks(monthly_summary["month"])
ax.set(title="Approved expenses by month", xlabel="Month", ylabel="KRW millions")
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1_000_000:.0f}"))
sns.despine()
fig.tight_layout()
fig.savefig("outputs/monthly_spend_en.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
department_totals = (
    approved.groupby("department", as_index=False)["amount_krw"]
    .sum()
    .sort_values("amount_krw", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=department_totals, x="amount_krw", y="department", color="#3978A8", ax=ax)
ax.set(title="Approved expenses by department", xlabel="KRW millions", ylabel="")
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1_000_000:.0f}"))
sns.despine()
fig.tight_layout()
fig.savefig("outputs/department_spend_en.png", dpi=180, bbox_inches="tight")
plt.show()

## 9. Reuse results for Korean output

In [ ]:
DEPARTMENT_KR = {"Marketing": "마케팅", "Sales": "영업", "Operations": "운영", "HR": "인사"}
CATEGORY_KR = {
    "Advertising": "광고비", "Software": "소프트웨어", "Travel": "출장비",
    "Meals": "식비", "Client Events": "고객행사", "Office Supplies": "사무용품",
    "Logistics": "물류비", "Training": "교육비", "Recruiting": "채용비",
}

department_summary_kr = department_summary.copy()
department_summary_kr["department"] = department_summary_kr["department"].replace(DEPARTMENT_KR)
department_summary_kr["category"] = department_summary_kr["category"].replace(CATEGORY_KR)
department_summary_kr = department_summary_kr.rename(columns={
    "department": "부서", "category": "항목", "amount_krw": "금액(원)"
})

print(department_summary_kr.head())

In [ ]:
department_totals_kr = department_totals.copy()
department_totals_kr["department"] = department_totals_kr["department"].replace(DEPARTMENT_KR)

plt.rc('font', family=['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'sans-serif'])


fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=department_totals_kr, x="amount_krw", y="department", color="#3978A8", ax=ax)
ax.set(title="부서별 승인 비용", xlabel="금액(백만원)", ylabel="")
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1_000_000:.0f}"))
sns.despine()
fig.tight_layout()
fig.savefig("outputs/department_spend_kr.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
## Save the cleaned and transformed expenses to one Excel
transformed_df.to_excel(output_dir / "clean_expenses.xlsx", index=False)

## Then save the summary of expenses to separate Excel files for English and Korean versions

with pd.ExcelWriter(output_dir / "expense_summary_en.xlsx") as writer:
    monthly_summary.to_excel(writer, sheet_name="Monthly", index=False)
    department_summary.to_excel(writer, sheet_name="Department category", index=False)

with pd.ExcelWriter(output_dir / "expense_summary_kr.xlsx") as writer:
    monthly_summary.rename(columns={"month": "월", "amount_krw": "금액(원)"}).to_excel(
        writer, sheet_name="월별", index=False
    )
    department_summary_kr.to_excel(writer, sheet_name="부서_항목", index=False)

## 10. Export this script to a Python file

Now that your Notebook is tested and working correctly, you can export it to a Python file for further use or deployment.

- Open the VS Code command palette (Ctrl+Shift+P or Cmd+Shift+P on Mac).
- Search for "Jupyter: Export to Python Script"
- A new editor tab will open with the exported Python script.
- Save the file to your desired location, for example `process_spendings.py`

Now delete all the output in the outputs folder and run the exported Python script to regenerate the outputs.

## Doom scenario

It's Friday 5:30pm and your manager walks into your office, pretty excited that he just received the monthly expense reports of June...

How long will it take you to add these numbers to the reports and charts?